### Differential Sharpe Ratio Approach

Implements proper online Sharpe ratio maximization with exponential decay.
This is theoretically grounded and fixes the EMA variance issues.

Reference: Moody & Saffell (2001) - "Learning to Trade via Direct Reinforcement"

In [2]:


import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from pathlib import Path
import json
import math

from stable_baselines3 import PPO, SAC, A2C
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback
import torch as th



In [3]:

class DifferentialSharpeEnv(gym.Env):
    """
    Portfolio environment using Differential Sharpe Ratio reward.
    
    Differential Sharpe is the gradient of the Sharpe ratio w.r.t. policy parameters.
    This provides cleaner gradients for RL optimization.
    """
    
    metadata = {"render_modes": []}
    
    def __init__(
        self,
        features_df: pd.DataFrame,
        returns_df: pd.DataFrame,
        tickers: list,
        feature_cols: list,
        softmax_temperature: float = 3.0,
        decay_factor: float = 0.95,
        random_start: bool = False,
    ):
        """
        Initialize Differential Sharpe environment.
        
        Parameters
        ----------
        decay_factor : float
            Exponential decay for online statistics (0.95 = ~20 step memory)
        """
        super().__init__()
        
        self.features_df = features_df
        self.returns_df = returns_df
        self.tickers = tickers
        self.feature_cols = feature_cols
        self.n_assets = len(tickers)
        self.n_features = len(feature_cols)
        self.temperature = softmax_temperature
        self.decay = decay_factor
        self.random_start = random_start
        
        # Validate
        assert 'ticker' in features_df.columns
        assert set(tickers).issubset(returns_df.columns)
        
        # Spaces
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(self.n_assets,), dtype=np.float32
        )
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.n_assets * self.n_features,), dtype=np.float32
        )
        
        self.dates = sorted(features_df.index.unique())
        
        # Episode state
        self._t = None
        self._weights = None
        
        # Differential Sharpe statistics
        self._mean_return = 0.0
        self._var_return = 1e-6
        self._n_effective = 0.0
        
    def _get_obs(self):
        """Get current observation."""
        date = self.dates[self._t]
        date_features = self.features_df.loc[[date]]
        
        feature_matrix = (
            date_features[date_features['ticker'].isin(self.tickers)]
            .set_index('ticker')[self.feature_cols]
            .reindex(self.tickers)
            .fillna(0.0)
            .values
            .astype(np.float32)
        )
        
        return feature_matrix.flatten()
    
    def _softmax(self, x):
        """Convert actions to portfolio weights."""
        x = np.asarray(x, dtype=np.float32) * self.temperature
        x = x - np.max(x)
        e = np.exp(x)
        return e / (e.sum() + 1e-12)
    
    def reset(self, seed=None, options=None):
        """Reset environment."""
        super().reset(seed=seed)
        
        # Starting position
        if self.random_start and len(self.dates) > 10:
            max_start = len(self.dates) - 5
            self._t = np.random.randint(1, max_start)
        else:
            self._t = 1
        
        # Initialize weights
        self._weights = np.ones(self.n_assets, dtype=np.float32) / self.n_assets
        
        # Reset Differential Sharpe statistics
        self._mean_return = 0.0
        self._var_return = 1e-6  # Small initial variance
        self._n_effective = 0.0
        
        return self._get_obs(), {}
    
    def step(self, action):
        """Execute one step with Differential Sharpe reward."""
        # Convert action to weights
        action = np.asarray(action, dtype=np.float32).reshape(-1)
        w = self._softmax(action)
        self._weights = w
        
        # Check if future return exists
        if self._t + 1 >= len(self.dates):
            return self._get_obs(), 0.0, True, False, {}
        
        # Get next period's returns
        next_date = self.dates[self._t + 1]
        r_vec = self.returns_df.loc[next_date, self.tickers].values.astype(np.float32)
        
        # Portfolio return (log return)
        port_log_r = float(np.dot(w, r_vec))
        
        # Update online statistics with exponential decay
        self._n_effective = 1.0 + self.decay * self._n_effective
        alpha = 1.0 / max(self._n_effective, 1.0)  # Adaptive learning rate
        
        delta = port_log_r - self._mean_return
        self._mean_return += alpha * delta
        self._var_return = self.decay * self._var_return + (1.0 - self.decay) * delta**2
        
        # Differential Sharpe Ratio reward
        std_return = math.sqrt(max(self._var_return, 1e-8))
        
        # Formula: A_t = (R_t * σ_t - 0.5 * μ_t * (R_t - μ_t)) / σ_t^2
        numerator = port_log_r * std_return - 0.5 * self._mean_return * delta
        denominator = std_return**2 + 1e-8
        
        diff_sharpe = numerator / denominator
        
        # Scale and clip
        reward = np.clip(diff_sharpe * math.sqrt(52), -10.0, 10.0)
        
        # Advance time
        self._t += 1
        terminated = self._t >= (len(self.dates) - 1)
        
        info = {
            'port_log_r': port_log_r,
            'mean_return': self._mean_return,
            'std_return': std_return,
            'sharpe_estimate': self._mean_return / std_return * math.sqrt(52) if std_return > 1e-8 else 0.0,
            'weights': w.copy()
        }
        
        return self._get_obs(), float(reward), terminated, False, info
    
    def run_full_pass(self, model):
        """Run full episode and return Sharpe ratio."""
        obs, _ = self.reset()
        returns = []
        done = False
        
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, done, _, info = self.step(action)
            if 'port_log_r' in info:
                returns.append(info['port_log_r'])
        
        returns = np.array(returns)
        if len(returns) < 2 or returns.std() < 1e-12:
            return 0.0
        
        return float((returns.mean() / returns.std()) * math.sqrt(52))


def create_differential_sharpe_env(data_dir: str, split: str, agent_type: str, **kwargs):
    """Create environment with Differential Sharpe reward."""
    data_dir = Path(data_dir)
    
    # Load data
    if agent_type == 'technical':
        features_path = data_dir / 'technical' / f'{split}.csv'
    else:
        features_path = data_dir / 'sentiment' / f'{split}.csv'
    
    features_df = pd.read_csv(features_path, index_col=0, parse_dates=True)
    returns_df = pd.read_csv(data_dir / f'returns_{split}.csv', index_col=0, parse_dates=True)
    
    # Load metadata
    with open(data_dir / 'metadata.json', 'r') as f:
        metadata = json.load(f)
    
    tickers = metadata['tickers']
    
    # Get indicator features only
    if agent_type == 'technical':
        feature_cols = metadata.get('technical_indicator_features',
                                     [c for c in metadata['technical_features'] 
                                      if c not in ['open', 'high', 'low', 'close', 'volume', 'return']])
    else:
        feature_cols = metadata.get('sentiment_indicator_features',
                                     [c for c in metadata['sentiment_features'] 
                                      if c not in ['open', 'high', 'low', 'close', 'volume', 'return']])
    
    feature_cols = [c for c in feature_cols if c in features_df.columns]
    
    env = DifferentialSharpeEnv(
        features_df=features_df,
        returns_df=returns_df,
        tickers=tickers,
        feature_cols=feature_cols,
        **kwargs
    )
    
    print(f"DifferentialSharpe-{agent_type.capitalize()}Env ({split}): {len(env.dates)} dates, {env.n_assets} assets, {env.n_features} features")
    
    return env


class ValidationCallback(BaseCallback):
    """Validation callback for early stopping."""
    
    def __init__(self, val_env, eval_freq: int = 5000, patience: int = 5,
                 save_path: str = None, verbose: int = 1):
        super().__init__(verbose)
        self.val_env = val_env
        self.eval_freq = eval_freq
        self.patience = patience
        self.save_path = save_path
        self.best_sharpe = -np.inf
        self.no_improve = 0
        
    def _on_step(self):
        if self.eval_freq <= 0 or (self.n_calls % self.eval_freq) != 0:
            return True
        
        val_sharpe = self.val_env.run_full_pass(self.model)
        
        if self.verbose:
            print(f"[Step {self.num_timesteps:,}] Val Sharpe: {val_sharpe:.3f} (Best: {self.best_sharpe:.3f})")
        
        if val_sharpe > self.best_sharpe + 1e-6:
            self.best_sharpe = val_sharpe
            self.no_improve = 0
            
            if self.save_path:
                self.model.save(self.save_path)
                if self.verbose:
                    print(f"  New best model saved")
        else:
            self.no_improve += 1
            if self.no_improve >= self.patience:
                if self.verbose:
                    print(f"  Early stopping")
                return False
        
        return True


def train_differential_sharpe(agent_type: str, algorithm: str, config: dict, verbose: bool = True):
    """Train agent with Differential Sharpe reward."""
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"Training {agent_type.upper()} Agent with {algorithm} (Differential Sharpe)")
        print(f"{'='*70}")
    
    # Create environments
    train_env = create_differential_sharpe_env(
        config['data_dir'], 'train', agent_type,
        softmax_temperature=config['softmax_temperature'],
        decay_factor=config['decay_factor'],
        random_start=True
    )
    val_env = create_differential_sharpe_env(
        config['data_dir'], 'val', agent_type,
        softmax_temperature=config['softmax_temperature'],
        decay_factor=config['decay_factor']
    )
    test_env = create_differential_sharpe_env(
        config['data_dir'], 'test', agent_type,
        softmax_temperature=config['softmax_temperature'],
        decay_factor=config['decay_factor']
    )
    
    vec_env = DummyVecEnv([lambda: train_env])
    
    # Create model
    model_kwargs = {
        'policy': 'MlpPolicy',
        'env': vec_env,
        'learning_rate': config['learning_rate'],
        'gamma': config['gamma'],
        'seed': config['seed'],
        'verbose': 0 if not verbose else 1,
    }
    
    if algorithm == 'PPO':
        model = PPO(
            **model_kwargs,
            n_steps=2048,
            batch_size=512,
            policy_kwargs=dict(activation_fn=th.nn.ReLU, net_arch=[256, 256]),
        )
    elif algorithm == 'SAC':
        model = SAC(
            **model_kwargs,
            buffer_size=200_000,
            batch_size=1024,
            policy_kwargs=dict(activation_fn=th.nn.ReLU, net_arch=dict(pi=[256, 256], qf=[256, 256]))
        )
    elif algorithm == 'A2C':
        model = A2C(
            **model_kwargs,
            n_steps=5,
            policy_kwargs=dict(activation_fn=th.nn.ReLU, net_arch=[256, 256])
        )
    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")
    
    # Training
    save_path = Path(config['models_dir']) / agent_type / 'differential_sharpe' / f"diff_sharpe_{agent_type}_{algorithm.lower()}.zip"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    callback = ValidationCallback(
        val_env=val_env,
        eval_freq=config['eval_freq'],
        patience=config['patience'],
        save_path=str(save_path),
        verbose=1 if verbose else 0
    )
    
    model.learn(total_timesteps=config['total_steps'], callback=callback)
    
    # Load best and evaluate
    if save_path.exists():
        if algorithm == 'PPO':
            model = PPO.load(str(save_path), env=vec_env)
        elif algorithm == 'SAC':
            model = SAC.load(str(save_path), env=vec_env)
        elif algorithm == 'A2C':
            model = A2C.load(str(save_path), env=vec_env)
    
    train_sharpe = train_env.run_full_pass(model)
    val_sharpe = val_env.run_full_pass(model)
    test_sharpe = test_env.run_full_pass(model)
    
    if verbose:
        print(f"\nResults:")
        print(f"  Train Sharpe: {train_sharpe:.3f}")
        print(f"  Val Sharpe:   {val_sharpe:.3f}")
        print(f"  Test Sharpe:  {test_sharpe:.3f}")
    
    return {
        'agent_type': agent_type,
        'algorithm': algorithm,
        'reward_type': 'differential_sharpe',
        'train_sharpe': train_sharpe,
        'val_sharpe': val_sharpe,
        'test_sharpe': test_sharpe,
        'model_path': str(save_path)
    }



In [ ]:


print("="*70)
print("DIFFERENTIAL SHARPE RATIO APPROACH")
print("="*70)

# Configuration
CONFIG = {
    'data_dir': 'data_hierarchical',
    'models_dir': 'models',
    'total_steps': 300_000,
    'eval_freq': 5_000,
    'patience': 5,
    'learning_rate': 3e-4,
    'gamma': 0.99,
    'softmax_temperature': 3.0,
    'decay_factor': 0.95,  # ~20 step memory
    'seed': 42,
}

Path(CONFIG['models_dir']).mkdir(exist_ok=True)

ALGORITHMS = ['PPO', 'SAC', 'A2C']

# Train Technical Agent
print("\n" + "="*70)
print("TRAINING TECHNICAL AGENT (Differential Sharpe)")
print("="*70)

tech_results = []
for algo in ALGORITHMS:
    result = train_differential_sharpe('technical', algo, CONFIG, verbose=True)
    tech_results.append(result)

tech_df = pd.DataFrame(tech_results).sort_values('val_sharpe', ascending=False)

print("\n" + "="*70)
print("TECHNICAL AGENT RESULTS")
print("="*70)
print(tech_df.to_string(index=False))

best_tech = tech_df.iloc[0].to_dict()
print(f"\nBest: {best_tech['algorithm']} (Test Sharpe: {best_tech['test_sharpe']:.3f})")

# Train Sentiment Agent
print("\n" + "="*70)
print("TRAINING SENTIMENT AGENT (Differential Sharpe)")
print("="*70)

sent_results = []
for algo in ALGORITHMS:
    result = train_differential_sharpe('sentiment', algo, CONFIG, verbose=True)
    sent_results.append(result)

sent_df = pd.DataFrame(sent_results).sort_values('val_sharpe', ascending=False)

print("\n" + "="*70)
print("SENTIMENT AGENT RESULTS")
print("="*70)
print(sent_df.to_string(index=False))

best_sent = sent_df.iloc[0].to_dict()
print(f"\nBest: {best_sent['algorithm']} (Test Sharpe: {best_sent['test_sharpe']:.3f})")

# Save results
results = {
    'approach': 'differential_sharpe',
    'technical': best_tech,
    'sentiment': best_sent,
    'config': CONFIG,
    'all_results': {
        'technical': tech_results,
        'sentiment': sent_results
    }
}

results_path = Path(CONFIG['models_dir']) / 'differential_sharpe_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"\n Results saved to {results_path}")

print("\n" + "="*70)
print("DIFFERENTIAL SHARPE TRAINING COMPLETE")
print("="*70)
print(f"\nTechnical: {best_tech['algorithm']} - Test Sharpe: {best_tech['test_sharpe']:.3f}")
print(f"Sentiment: {best_sent['algorithm']} - Test Sharpe: {best_sent['test_sharpe']:.3f}")

return results
